In [86]:
!pip install pandas

You should consider upgrading via the '/usr/bin/python3 -m pip install --upgrade pip' command.


In [87]:
import os
import pandas as pd
import numpy as np
#import networkx as nx
#import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [88]:
import h5py

In [89]:
hidden_units = [32, 32]
learning_rate = 0.01
dropout_rate = 0.5
num_epochs = 2
batch_size = 8 #256


In [90]:
def compile_model(model):
    # Compile the model.
    model.compile(
        optimizer="rmsprop",
        #optimizer=keras.optimizers.Adam(learning_rate),
        loss="binary_crossentropy",
        # Tati: categorical_crossentropy, expects the labels to follow a categorical encoding. 
        #       With integer labels, you should use sparse_categorical_crossentropy.
        #       This new loss function is still mathematically the same as categorical_crossentropy; it just has a different interface.
        #metrics=[keras.metrics.SparseCategoricalAccuracy(name="acc")],
    )
    # Create an early stopping callback.
    #early_stopping = keras.callbacks.EarlyStopping(
    #    monitor="val_acc", patience=50, restore_best_weights=True
    #
    #)
    return model

# This function trains an input model using the given training data.
def run_experiment(model, x_train, y_train): 
    guardarModelo = keras.callbacks.ModelCheckpoint(
        filepath="/mnt/pesos/",  # "checkpoint_path.keras",
        monitor="val_loss",
        save_best_only=True,
        save_format="tf",
    )
    # Fit the model.
    history = model.fit(
        x=x_train,
        y=y_train,
        epochs=num_epochs,
        batch_size=batch_size,
        validation_split=0.15,
        callbacks=[guardarModelo], #early_stopping],
    )

    return history


def create_ffn(hidden_units, dropout_rate, name=None):
    fnn_layers = []

    for units in hidden_units:
        fnn_layers.append(layers.BatchNormalization())
        fnn_layers.append(layers.Dropout(dropout_rate))
        fnn_layers.append(layers.Dense(units, activation=tf.nn.gelu))

    return keras.Sequential(fnn_layers, name=name)


In [91]:
class GraphConvLayer(layers.Layer):
    def __init__(
        self,
        hidden_units,
        dropout_rate=0.2,
        aggregation_type="mean",
        combination_type="concat",
        normalize=False,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        self.aggregation_type = aggregation_type
        self.combination_type = combination_type
        self.normalize = normalize

        self.ffn_prepare = create_ffn(hidden_units, dropout_rate)
        if self.combination_type == "gated":
            self.update_fn = layers.GRU(
                units=hidden_units,
                activation="tanh",
                recurrent_activation="sigmoid",
                dropout=dropout_rate,
                return_state=True,
                recurrent_dropout=dropout_rate,
            )
        else:
            self.update_fn = create_ffn(hidden_units, dropout_rate)

    def prepare(self, node_repesentations, weights=None):
        # node_repesentations shape is [num_edges, embedding_dim].
        messages = self.ffn_prepare(node_repesentations)
        if weights is not None:
            messages = messages * tf.expand_dims(weights, -1)
        return messages

    def aggregate(self, node_indices, neighbour_messages, node_repesentations):
        # node_indices shape is [num_edges].
        # neighbour_messages shape: [num_edges, representation_dim].
        # node_repesentations shape is [num_nodes, representation_dim]
        num_nodes = node_repesentations.shape[0]
        if self.aggregation_type == "sum":
            aggregated_message = tf.math.unsorted_segment_sum(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "mean":
            aggregated_message = tf.math.unsorted_segment_mean(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        elif self.aggregation_type == "max":
            aggregated_message = tf.math.unsorted_segment_max(
                neighbour_messages, node_indices, num_segments=num_nodes
            )
        else:
            raise ValueError(f"Invalid aggregation type: {self.aggregation_type}.")

        return aggregated_message

    def update(self, node_repesentations, aggregated_messages):
        # node_repesentations shape is [num_nodes, representation_dim].
        # aggregated_messages shape is [num_nodes, representation_dim].
        if self.combination_type == "gru":
            # Create a sequence of two elements for the GRU layer.
            h = tf.stack([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "concat":
            # Concatenate the node_repesentations and aggregated_messages.
            h = tf.concat([node_repesentations, aggregated_messages], axis=1)
        elif self.combination_type == "add":
            # Add node_repesentations and aggregated_messages.
            h = node_repesentations + aggregated_messages
        else:
            raise ValueError(f"Invalid combination type: {self.combination_type}.")

        # Apply the processing function.
        node_embeddings = self.update_fn(h)
        if self.combination_type == "gru":
            node_embeddings = tf.unstack(node_embeddings, axis=1)[-1]

        if self.normalize:
            node_embeddings = tf.nn.l2_normalize(node_embeddings, axis=-1)
        return node_embeddings

    def call(self, inputs):
        """Process the inputs to produce the node_embeddings.

        inputs: a tuple of three elements: node_repesentations, edges, edge_weights.
        Returns: node_embeddings of shape [num_nodes, representation_dim].
        """

        node_repesentations, edges, edge_weights = inputs
        # Get node_indices (source) and neighbour_indices (target) from edges.
        node_indices, neighbour_indices = edges[0], edges[1]
        # neighbour_repesentations shape is [num_edges, representation_dim].
        neighbour_repesentations = tf.gather(node_repesentations, neighbour_indices)

        # Prepare the messages of the neighbours.
        neighbour_messages = self.prepare(neighbour_repesentations, edge_weights)
        # Aggregate the neighbour messages.
        aggregated_messages = self.aggregate(
            node_indices, neighbour_messages, node_repesentations
        )
        # Update the node embedding with the neighbour messages.
        return self.update(node_repesentations, aggregated_messages)


In [92]:
class GNNNodeClassifier(tf.keras.Model):
    def __init__(
        self,
        graph_info,
        num_classes,
        hidden_units,
        aggregation_type="sum",
        combination_type="concat",
        dropout_rate=0.2,
        normalize=True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        # Unpack graph_info to three elements: node_features, edges, and edge_weight.
        node_features, edges, edge_weights = graph_info
        self.node_features = node_features
        self.edges = edges
        self.edge_weights = edge_weights
        # Set edge_weights to ones if not provided.
        if self.edge_weights is None:
            self.edge_weights = tf.ones(shape=edges.shape[1])
        # Scale edge_weights to sum to 1.
        self.edge_weights = self.edge_weights / tf.math.reduce_sum(self.edge_weights)

        # Create a process layer.
        self.preprocess = create_ffn(hidden_units, dropout_rate, name="preprocess")
        # Create the first GraphConv layer.
        self.conv1 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv1",
        )
        # Create the second GraphConv layer.
        self.conv2 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv2",
        )
        # Create a postprocess layer.
        self.postprocess = create_ffn(hidden_units, dropout_rate, name="postprocess")
        # Create a compute logits layer.
        self.compute_logits = layers.Dense(units=num_classes, name="logits")  
          # Tati: For Tensorflow: logits is a name that it is thought to imply that this Tensor is the quantity that is being mapped to probabilities by the Softmax
    
    def load_weights(self, model_path):
        self.model = keras.models.load_weights(model_path, by_name=False)
        
    def call(self, input_node_indices):
        # Preprocess the node_features to produce node representations.
        x = self.preprocess(self.node_features)
        # Apply the first graph conv layer.
        x1 = self.conv1((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x1 + x
        # Apply the second graph conv layer.
        x2 = self.conv2((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x2 + x
        # Postprocess node embedding.
        x = self.postprocess(x)
        # Fetch node embeddings for the input node_indices.
        node_embeddings = tf.gather(x, input_node_indices)
        # Compute logits
        return self.compute_logits(node_embeddings)


In [95]:
# Training
training_grafos = pd.read_csv(
    "/mnt/edges_int/edges_INT_capture20110810.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features = pd.read_csv(
    "/mnt/features_int/features_INT_capture20110810.csv",
    sep=",",  
    header=0
)

# Create an edges array (adjacency matrix) of shape [2, num_edges]
training_edges = training_grafos[["source", "target"]].to_numpy().T
#validation_edges = validation_grafos[["source", "target"]].to_numpy().T

# Create an edge weights array.
training_edge_weights = training_grafos[["weight"]].to_numpy().T 
training_edge_weights = training_edge_weights.reshape((training_edges.shape[-1],))
training_edge_weights = tf.convert_to_tensor(training_edge_weights)

# Create a node features array of shape [num_nodes, num_features].
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)

# Create graph info tuple with node_features, edges, and edge_weights.
graph_info = (training_node_features, training_edges, training_edge_weights)

print("Edges shape:", training_edges.shape)
print("Nodes shape:", training_node_features.shape)


Edges shape: (2, 1234211)
Nodes shape: (605195, 4)


<ipython-input-95-3791985ae8ec>:27: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32


In [96]:
## GNN
gnn_model = GNNNodeClassifier(
    graph_info=graph_info,
    num_classes=2,
    hidden_units=hidden_units,
    dropout_rate=dropout_rate,
    name="gnn_model",
)

print("GNN output shape:", gnn_model([1, 10, 100]))

print(gnn_model.summary())

modelo = compile_model(gnn_model)


train_data = training_features.sample(frac=1)
#test_data = validation_features.sample(frac=1)

print("Train data shape:", train_data.shape) 
#print("Test data shape:", test_data.shape) 

# Create train and test features as a numpy array.
x_train = train_data[feature_names].to_numpy()
#x_test = test_data[feature_names].to_numpy()
# Create train and test targets as a numpy array.
y_train = train_data["label"]
#y_test = test_data["label"]


x_train = train_data.node.to_numpy()
print("x_train.dtype = ", x_train.dtype)
print("y_train.dtype = ", y_train.dtype)

history = run_experiment(modelo, x_train, y_train)


GNN output shape: tf.Tensor(
[[-0.2522846  -0.6925785 ]
 [-0.14616998 -0.12014393]
 [-0.14616998 -0.12014393]], shape=(3, 2), dtype=float32)
Model: "gnn_model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 preprocess (Sequential)     (605195, 32)              1360      
                                                                 
 graph_conv1 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 graph_conv2 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 postprocess (Sequential)    (605195, 32)              2368      
                                                                 
 logits (Dense)              multiple           

<ipython-input-96-ce64d62ff60a>:24: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  x_train = train_data[feature_names].to_numpy()


53596/64302 [========================>.....] - ETA: 1:24:20 - loss: 0.0088

KeyboardInterrupt: 

In [70]:
modelo.save_weights('/mnt/pesos/mis_pesos.h5')

In [71]:
os.listdir("/mnt/pesos")

['mis_pesos.h5',
 'saved_model.pb',
 'mis_pesos.index',
 'mis_pesos.data-00000-of-00001',
 'variables',
 'assets',
 'checkpoint',
 'keras_metadata.pb']

In [72]:
class GNNNodeClassifier2(tf.keras.Model):
    def __init__(
        self,
        graph_info,
        num_classes,
        hidden_units,
        aggregation_type="sum",
        combination_type="concat",
        dropout_rate=0.2,
        normalize=True,
        *args,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)

        # Unpack graph_info to three elements: node_features, edges, and edge_weight.
        node_features, edges, edge_weights = graph_info
        self.node_features = node_features
        self.edges = edges
        self.edge_weights = edge_weights
        # Set edge_weights to ones if not provided.
        if self.edge_weights is None:
            self.edge_weights = tf.ones(shape=edges.shape[1])
        # Scale edge_weights to sum to 1.
        self.edge_weights = self.edge_weights / tf.math.reduce_sum(self.edge_weights)

        # Create a process layer.
        self.preprocess = create_ffn(hidden_units, dropout_rate, name="preprocess2")
        # Create the first GraphConv layer.
        self.conv1 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv1",
        )
        # Create the second GraphConv layer.
        self.conv2 = GraphConvLayer(
            hidden_units,
            dropout_rate,
            aggregation_type,
            combination_type,
            normalize,
            name="graph_conv2",
        )
        # Create a postprocess layer.
        self.postprocess = create_ffn(hidden_units, dropout_rate, name="postprocess2")
        # Create a compute logits layer.
        self.compute_logits = layers.Dense(units=num_classes, name="logits2")  
          # Tati: For Tensorflow: logits is a name that it is thought to imply that this Tensor is the quantity that is being mapped to probabilities by the Softmax
    
    def load_weights(self, weights_path):
        self.load_weights(weights_path)
        
    def call(self, input_node_indices):
        # Preprocess the node_features to produce node representations.
        x = self.preprocess(self.node_features)
        # Apply the first graph conv layer.
        x1 = self.conv1((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x1 + x
        # Apply the second graph conv layer.
        x2 = self.conv2((x, self.edges, self.edge_weights))
        # Skip connection.
        x = x2 + x
        # Postprocess node embedding.
        x = self.postprocess(x)
        # Fetch node embeddings for the input node_indices.
        node_embeddings = tf.gather(x, input_node_indices)
        # Compute logits
        return self.compute_logits(node_embeddings)


In [73]:
# Training
training_grafos = pd.read_csv(
    "/mnt/edges_int/edges_INT_capture20110815-2.ncol",
    sep=" ",  
    header=None,  # no heading row
    names=["source", "target", "weight"],  # set our own names for the columns
)

training_features = pd.read_csv(
    "/mnt/features_int/features_INT_capture20110815-2.csv",
    sep=",",  
    header=0
)

# Create an edges array (adjacency matrix) of shape [2, num_edges]
training_edges = training_grafos[["source", "target"]].to_numpy().T
#validation_edges = validation_grafos[["source", "target"]].to_numpy().T

# Create an edge weights array.
training_edge_weights = training_grafos[["weight"]].to_numpy().T 
training_edge_weights = training_edge_weights.reshape((training_edges.shape[-1],))
training_edge_weights = tf.convert_to_tensor(training_edge_weights)

# Create a node features array of shape [num_nodes, num_features].
feature_names = set(training_features.columns) - {"node", "label"}
training_node_features = tf.cast(
    training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32
)

# Create graph info tuple with node_features, edges, and edge_weights.
graph_info2 = (training_node_features, training_edges, training_edge_weights)

print("Edges shape:", training_edges.shape)
print("Nodes shape:", training_node_features.shape)


Edges shape: (2, 85782)
Nodes shape: (41399, 4)


<ipython-input-73-e153b84aea06>:27: FutureWarning: Passing a set as an indexer is deprecated and will raise in a future version. Use a list instead.
  training_features.sort_values("node")[feature_names].to_numpy(), dtype=tf.dtypes.float32


In [74]:
## GNN
gnn_model2 = GNNNodeClassifier2(
    graph_info=graph_info2,
    num_classes=2,
    hidden_units=hidden_units,
    dropout_rate=dropout_rate,
    name="gnn_model2",
)

print("GNN output shape:", gnn_model2([1, 10, 100]))

print(gnn_model2.summary())


modelo2 = compile_model(gnn_model2)
modelo2.load_weights("/mnt/pesos/mis_pesos", by_name=True)

train_data = training_features.sample(frac=1)
#test_data = validation_features.sample(frac=1)

print("Train data shape:", train_data.shape) 
#print("Test data shape:", test_data.shape) 

# Create train and test features as a numpy array.
x_train = train_data[feature_names].to_numpy()
#x_test = test_data[feature_names].to_numpy()
# Create train and test targets as a numpy array.
y_train = train_data["label"]
#y_test = test_data["label"]


x_train = train_data.node.to_numpy()
print("x_train.dtype = ", x_train.dtype)
print("y_train.dtype = ", y_train.dtype)

history2 = run_experiment(modelo2, x_train, y_train)


GNN output shape: tf.Tensor(
[[ 0.03602814 -0.38242173]
 [-0.10077536 -0.05550982]
 [-0.10077536 -0.05550982]], shape=(3, 2), dtype=float32)
Model: "gnn_model2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 preprocess2 (Sequential)    (41399, 32)               1360      
                                                                 
 graph_conv1 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 graph_conv2 (GraphConvLayer  multiple                 5888      
 )                                                               
                                                                 
 postprocess2 (Sequential)   (41399, 32)               2368      
                                                                 
 logits2 (Dense)             multiple          

TypeError: load_weights() got an unexpected keyword argument 'by_name'